In [22]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import joblib
import os

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

df = pd.read_parquet(
    "../Data/processed/imd_temperature_2010_2025.parquet"
)

df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(
    ["latitude", "longitude", "date"]
).reset_index(drop=True)

print(df.shape)
print(df.head())

(2074620, 4)
        date  latitude  longitude  max_temperature
0 2010-01-01       8.5       73.5        32.869999
1 2010-01-02       8.5       73.5        32.520000
2 2010-01-03       8.5       73.5        32.410000
3 2010-01-04       8.5       73.5        31.860001
4 2010-01-05       8.5       73.5        32.020000


In [23]:
group_cols = ["latitude", "longitude"]

# Calendar features
df["month"] = df["date"].dt.month
df["day_of_year"] = df["date"].dt.dayofyear

df["month_sin"] = np.sin(
    2 * np.pi * df["day_of_year"] / 365.25
)

df["month_cos"] = np.cos(
    2 * np.pi * df["day_of_year"] / 365.25
)

# Lag features
df["temp_lag_1"] = (
    df.groupby(group_cols)["max_temperature"].shift(1)
)

df["temp_lag_2"] = (
    df.groupby(group_cols)["max_temperature"].shift(2)
)

df["temp_lag_3"] = (
    df.groupby(group_cols)["max_temperature"].shift(3)
)

df["temp_lag_7"] = (
    df.groupby(group_cols)["max_temperature"].shift(7)
)

# Rolling features - only past values
df["temp_rolling_mean_3"] = (
    df.groupby(group_cols)["max_temperature"]
      .transform(
          lambda x: x.shift(1).rolling(3).mean()
      )
)

df["temp_rolling_mean_7"] = (
    df.groupby(group_cols)["max_temperature"]
      .transform(
          lambda x: x.shift(1).rolling(7).mean()
      )
)

df["temp_rolling_std_7"] = (
    df.groupby(group_cols)["max_temperature"]
      .transform(
          lambda x: x.shift(1).rolling(7).std()
      )
)

# Temperature changes
df["temperature_change_1d"] = (
    df["temp_lag_1"] - df["temp_lag_2"]
)

df["temperature_change_3d"] = (
    df["temp_lag_1"] - df["temp_lag_3"]
)

In [24]:
train_mask = df["date"].dt.year <= 2022

train_df = df.loc[train_mask].copy()

In [25]:
monthly_baseline = (
    train_df.groupby(
        ["latitude", "longitude", "month"]
    )["max_temperature"]
    .mean()
    .reset_index(name="monthly_baseline")
)

df = df.merge(
    monthly_baseline,
    on=["latitude", "longitude", "month"],
    how="left"
)

df["temperature_anomaly"] = (
    df["max_temperature"] - df["monthly_baseline"]
)

df["anomaly_lag_1"] = (
    df["temp_lag_1"] - df["monthly_baseline"]
)

In [26]:
df["target_temperature"] = (
    df.groupby(group_cols)["max_temperature"].shift(-1)
)

In [27]:
features = [
    "latitude",
    "longitude",
    "month",
    "day_of_year",
    "month_sin",
    "month_cos",
    "max_temperature",
    "temp_lag_1",
    "temp_lag_2",
    "temp_lag_3",
    "temp_lag_7",
    "temp_rolling_mean_3",
    "temp_rolling_mean_7",
    "temp_rolling_std_7",
    "temperature_change_1d",
    "temperature_change_3d",
    "monthly_baseline",
    "temperature_anomaly",
    "anomaly_lag_1"
]

In [28]:
forecast_df = df[
    ["date"] + features + ["target_temperature"]
].dropna(
    subset=features + ["target_temperature"]
).copy()

print(forecast_df.shape)
print(forecast_df.columns.tolist())

(2055334, 21)
['date', 'latitude', 'longitude', 'month', 'day_of_year', 'month_sin', 'month_cos', 'max_temperature', 'temp_lag_1', 'temp_lag_2', 'temp_lag_3', 'temp_lag_7', 'temp_rolling_mean_3', 'temp_rolling_mean_7', 'temp_rolling_std_7', 'temperature_change_1d', 'temperature_change_3d', 'monthly_baseline', 'temperature_anomaly', 'anomaly_lag_1', 'target_temperature']


In [29]:
os.makedirs("../Data/processed", exist_ok=True)

forecast_df.to_parquet(
    "../Data/processed/temperature_forecasting.parquet2",
    index=False
)

print("Forecasting dataset saved.")

Forecasting dataset saved.


In [30]:
train_mask = forecast_df["date"].dt.year <= 2022
val_mask = forecast_df["date"].dt.year == 2023
test_mask = forecast_df["date"].dt.year >= 2024

X_train = forecast_df.loc[train_mask, features]
y_train = forecast_df.loc[train_mask, "target_temperature"]

X_val = forecast_df.loc[val_mask, features]
y_val = forecast_df.loc[val_mask, "target_temperature"]

X_test = forecast_df.loc[test_mask, features]
y_test = forecast_df.loc[test_mask, "target_temperature"]

print(X_train.shape, X_val.shape, X_test.shape)

(1668083, 19) (129138, 19) (258113, 19)


In [31]:
lgb_forecast = lgb.LGBMRegressor(
    objective="regression",
    n_estimators=2000,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=50,
    subsample=0.85,
    subsample_freq=1,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=0.5,
    random_state=42,
    n_jobs=-1
)

lgb_forecast.fit(
    X_train,
    y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    eval_names=["train", "validation"],
    callbacks=[
        lgb.early_stopping(100, verbose=True),
        lgb.log_evaluation(100)
    ]
)

c:\Users\Qudsiya Siddique\Desktop\Climat\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.008574 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4140
[LightGBM] [Info] Number of data points in the train set: 1668083, number of used features: 19
[LightGBM] [Info] Start training from score 30.975245
Training until validation scores don't improve for 100 rounds
[100]	train's l2: 1.58347	validation's l2: 1.72679
[200]	train's l2: 1.42702	validation's l2: 1.67067
Early stopping, best iteration is:
[156]	train's l2: 1.45991	validation's l2: 1.66677


,num_leaves,63
,learning_rate,0.03
,n_estimators,2000
,objective,'regression'
,min_child_samples,50
,subsample,0.85
,subsample_freq,1
,colsample_bytree,0.85
,reg_alpha,0.1
,reg_lambda,0.5
,random_state,42


In [32]:
val_pred = lgb_forecast.predict(
    X_val,
    num_iteration=lgb_forecast.best_iteration_
)

test_pred = lgb_forecast.predict(
    X_test,
    num_iteration=lgb_forecast.best_iteration_
)

print("VALIDATION")
print("MAE :", mean_absolute_error(y_val, val_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_val, val_pred)))
print("R²  :", r2_score(y_val, val_pred))

print("\nTEST")
print("MAE :", mean_absolute_error(y_test, test_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, test_pred)))
print("R²  :", r2_score(y_test, test_pred))

VALIDATION
MAE : 0.9051276083952184
RMSE: 1.2910331816871585
R²  : 0.9461983674440982

TEST
MAE : 0.9222516017588481
RMSE: 1.2781364613521584
R²  : 0.9481891121841661


In [33]:
importance = (
    pd.Series(
        lgb_forecast.feature_importances_,
        index=features
    )
    .sort_values(ascending=False)
)

print(importance)

max_temperature          2335
anomaly_lag_1            1506
temp_lag_1                942
month_cos                 726
monthly_baseline          717
temperature_anomaly       634
month_sin                 625
day_of_year               538
temperature_change_3d     387
latitude                  243
longitude                 225
temp_rolling_std_7        163
temp_lag_7                160
temp_rolling_mean_7       143
month                      97
temperature_change_1d      91
temp_rolling_mean_3        49
temp_lag_3                 46
temp_lag_2                 45
dtype: int32


In [34]:
os.makedirs("../models", exist_ok=True)

joblib.dump(
    lgb_forecast,
    "../models/temperature_forecasting_lgbm.joblib"
)

joblib.dump(
    {
        "features": features,
        "best_iteration": lgb_forecast.best_iteration_
    },
    "../models/temperature_forecasting_metadata.joblib"
)

print("Model saved successfully.")

Model saved successfully.
